# Strict AOG v42 Template Diagnostics

This notebook diagnoses the strict AOG v42 template library. It reuses the revised v42 diagnostic predictions, scores template usage and simplicity, adds the Strict AOG Class Templates layout view from the occlusion notebook, creates validation-image overlays, and writes an interpretation markdown file.


## 0. Setup


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / '.git').exists() and (p / 'experiments').exists():
            return p
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_ROOT = find_repo_root()
EXP = REPO_ROOT / 'experiments' / 'aog_v19_v23'
RUNS = EXP / 'runs'
SRC = EXP / 'src'
SCRIPTS = EXP / 'scripts'
for p in [SRC, SCRIPTS]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

GRAMMAR = RUNS / 'strict_aog_cache_v42' / 'strict_aog_v42.pt'
DIAG = RUNS / 'strict_aog_v42_layout_rich_view_capped_templates_diagnostics'
OUT = RUNS / 'strict_aog_v42_template_diagnostics'
CONFIG = REPO_ROOT / 'configs' / 'stage1_quality_upgrade.yaml'
PREFIX = 'strict_aog_v42'
OUT.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([{'repo_root': str(REPO_ROOT), 'grammar': str(GRAMMAR), 'diagnostic_source': str(DIAG), 'out_dir': str(OUT)}]))


## 1. Resolve Existing v42 Diagnostic Predictions

The analysis uses the prediction table produced by `strict_aog_v42_revised_diagnostics.ipynb`, so the template usage metrics match the same validation split and template assignments.


In [ ]:
def resolve_diag(source):
    source = Path(source)
    if source.is_dir():
        return source
    zip_path = source if source.suffix == '.zip' else source.with_suffix('.zip')
    if zip_path.exists():
        dest = OUT / '_expanded_revised_diagnostics'
        if not dest.exists():
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(dest)
        return dest
    raise FileNotFoundError(source)

base = resolve_diag(DIAG)
pred_csv = next((p for p in [base / 'tables' / 'predictions.csv', base / 'predictions.csv'] if p.exists()), None)
if pred_csv is None:
    raise FileNotFoundError('Could not find predictions.csv under the revised diagnostics output')
pred = pd.read_csv(pred_csv)
acc = float(pred['correct'].astype(bool).mean()) if 'correct' in pred else np.nan
display(pd.DataFrame([{'predictions_csv': str(pred_csv), 'rows': len(pred), 'accuracy': acc, 'classes': pred['true'].nunique(), 'true_templates': pred['true_template'].nunique()}]))
display(pred.head())


## 2. Template Quality Analysis

This runs the committed quality script to measure template usage, simplicity, relation density, duplicate slots, high-use complex templates, and high-use rigid templates.


In [ ]:
quality_script = SCRIPTS / 'analyze_strict_aog_template_quality.py'
for required in [GRAMMAR, quality_script, pred_csv]:
    if not Path(required).exists():
        raise FileNotFoundError(required)
cmd = [sys.executable, str(quality_script), '--grammar', str(GRAMMAR), '--predictions', str(pred_csv), '--out-dir', str(OUT), '--prefix', PREFIX]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

template_csv = OUT / f'{PREFIX}_template_quality_by_template.csv'
class_csv = OUT / f'{PREFIX}_template_quality_by_class.csv'
report_md = OUT / f'{PREFIX}_template_quality_report.md'
template_df = pd.read_csv(template_csv)
class_df = pd.read_csv(class_csv)
display(class_df.sort_values(['max_usage_frac', 'num_low_usage_templates'], ascending=[False, False]))
display(template_df.sort_values(['true_usage_frac', 'num_slots'], ascending=[False, False]).head(15))
if report_md.exists():
    display(Markdown(report_md.read_text(encoding='utf-8')))


## 3. Strict AOG Class Templates

This is the class-template layout visualization used in the occlusion notebook's `Strict AOG Class Templates` chunk, routed through the shared helper script so the notebook and script stay consistent.


In [ ]:
from partcat_hkg.strict_aog.grammar import load_strict_aog
from summarize_strict_aog_templates import plot_class_templates, template_summary_tables

grammar = load_strict_aog(GRAMMAR)
template_summary, template_edges = template_summary_tables(grammar)
class_template_summary_csv = OUT / f'{PREFIX}_class_template_summary.csv'
class_template_edges_csv = OUT / f'{PREFIX}_class_template_edges.csv'
class_template_fig = OUT / f'{PREFIX}_class_templates.png'
template_summary.to_csv(class_template_summary_csv, index=False)
template_edges.to_csv(class_template_edges_csv, index=False)
plot_class_templates(grammar, class_template_fig)
display(template_summary.sort_values(['template_issues', 'num_edges', 'num_slots'], ascending=[False, False, False]).head(20))
display(Image(filename=str(class_template_fig)))


## 4. Template Overlay Examples

The class-template plot shows abstract template geometry. These overlay pages put the same slots and relation edges over concrete validation images selected from the revised diagnostic predictions.


In [ ]:
overlay_script = SCRIPTS / 'visualize_strict_aog_template_overlays.py'
for required in [CONFIG, overlay_script, GRAMMAR, pred_csv, template_csv]:
    if not Path(required).exists():
        raise FileNotFoundError(required)
overlay_dir = OUT / 'template_overlay_examples'
cmd = [sys.executable, str(overlay_script), '--config', str(CONFIG), '--grammar', str(GRAMMAR), '--predictions', str(pred_csv), '--template-quality', str(template_csv), '--out-dir', str(overlay_dir), '--prefix', PREFIX, '--examples-per-template', '3', '--image-size', '320']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
overlay_summary_csv = overlay_dir / f'{PREFIX}_template_overlay_summary.csv'
overlay_index = overlay_dir / f'{PREFIX}_template_overlay_index.md'
overlay_summary = pd.read_csv(overlay_summary_csv)
display(overlay_summary.head(20))
display(Markdown(overlay_index.read_text(encoding='utf-8')))


## 5. Interpretation

This cell reads the generated tables and visualization summaries, then writes `strict_aog_v42_template_diagnostic_interpretation.md`.


In [ ]:
template_df = pd.read_csv(template_csv)
class_df = pd.read_csv(class_csv)
template_summary = pd.read_csv(class_template_summary_csv)
overlay_summary = pd.read_csv(overlay_summary_csv) if overlay_summary_csv.exists() else pd.DataFrame()
flags = template_df['quality_flags'].fillna('').astype(str)
flagged = template_df[flags != ''].copy()
dominant = template_df.sort_values('true_usage_frac', ascending=False).head(8).copy()
collapsed = class_df.sort_values('max_usage_frac', ascending=False).head(6).copy()
low_usage_total = int((template_df['true_usage_frac'] < 0.05).sum())
complex_total = int(flags.str.contains('high_use_complex_template', regex=False).sum())
rigid_total = int(flags.str.contains('high_use_rigid_layout', regex=False).sum())
edge_issue_count = int(template_summary['template_issues'].fillna('').astype(str).ne('').sum()) if 'template_issues' in template_summary else 0
acc = float(pred['correct'].astype(bool).mean())
lines = ['# Strict AOG v42 Template Diagnostic Interpretation', '', '- validation rows read from revised diagnostics: {}'.format(len(pred)), '- validation accuracy in the prediction table: {:.4f}'.format(acc), '- templates scored: {} across {} classes'.format(len(template_df), class_df['class_name'].nunique()), '- low-use templates below 5 percent of their class: {}'.format(low_usage_total), '- flagged high-use complex templates: {}'.format(complex_total), '- flagged high-use rigid templates: {}'.format(rigid_total), '- templates with structural layout issues in the class-template view: {}'.format(edge_issue_count), '', '## Main Reading', '', 'The first check is whether v42 uses its templates as real layout modes rather than decorative capacity. If max_usage_frac is very high for a class, that class is effectively represented by one dominant template. If effective_templates_used is close to the prior count, the class is using multiple templates in a healthier way.', '', 'The class-template figure is the structural sanity check. Good templates show slots clustered around recognizable part layouts, with relation edges supporting those slots instead of becoming a dense mesh. Dense edges, repeated singleton parts, or low endpoint coverage suggest that the grammar may be encoding noise or duplicated parts rather than clean compositional structure.', '', 'The overlay pages are the visual sanity check. If the boxes align with visible parts in real examples, the template is interpretable and likely useful. If a high-use template repeatedly puts slots on background, wrong part regions, or occluded evidence, rebuild it even if the usage fraction is high.', '', '## Highest-usage templates', '', '```', dominant[['class_name', 'template_id', 'true_usage_frac', 'accuracy_when_true_template', 'num_slots', 'num_edges', 'position_std_mean', 'quality_flags']].to_string(index=False), '```', '', '## Classes most concentrated on one template', '', '```', collapsed[['class_name', 'accuracy', 'effective_templates_used', 'effective_templates_prior', 'max_usage_frac', 'num_low_usage_templates']].to_string(index=False), '```', '', '## Actionable template flags', '']
if flagged.empty:
    lines.append('No templates were flagged by the usage and simplicity heuristic. Inspect the overlay pages for remaining qualitative issues.')
else:
    lines += ['```', flagged[['class_name', 'template_id', 'true_usage_frac', 'accuracy_when_true_template', 'num_slots', 'num_edges', 'max_degree', 'position_std_mean', 'quality_flags']].sort_values(['true_usage_frac', 'num_slots'], ascending=[False, False]).to_string(index=False), '```']
lines += ['', '## Practical next step', '', 'Start with templates that are both high-use and flagged. If they are complex but visually aligned, they are probably important canonical poses. If they are complex and poorly aligned, simplify the slot set or rebuild the clustering. Low-use templates should be merged or pruned unless their overlay pages reveal a rare but meaningful pose.']
interpretation = chr(10).join(lines)
interp_path = OUT / f'{PREFIX}_template_diagnostic_interpretation.md'
interp_path.write_text(interpretation, encoding='utf-8')
display(Markdown(interpretation))
print(interp_path)


## 6. Artifact Index


In [ ]:
rows = []
for root in [OUT, OUT / 'template_overlay_examples']:
    if root.exists():
        for path in sorted(root.glob('*')):
            if path.is_file():
                rows.append({'artifact': path.name, 'path': str(path), 'bytes': path.stat().st_size})
display(pd.DataFrame(rows))
